# AGENT WEB · Colab Blender 4.x
Google ColabのGPUランタイム（L4またはT4）でBlenderレンダリングを実行するための接続用ノートブックです。
AGENT WEBからの要求を受け取り、Cycles GPUまたはEeveeによる静止画・アニメーションレンダリングを行います。

In [ ]:
# 1. Blender Linux 4.2 LTS のセットアップ
!apt-get -qq update && apt-get -qq install -y libgl1 libegl1 libxi6 libxkbcommon0
!if [ ! -d "/content/blender-4.2.3-linux-x64" ]; then \
    wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz && \
    tar -xf blender-4.2.3-linux-x64.tar.xz -C /content/; \
fi
!ln -sf /content/blender-4.2.3-linux-x64/blender /usr/local/bin/blender
!blender --version

In [ ]:
# 2. GPU (CUDA/OptiX) 有効化の検証
import subprocess
enable_gpu_script = """
import bpy
prefs = bpy.context.preferences.addons['cycles'].preferences
prefs.get_devices()
for dev_type in ('OPTIX', 'CUDA'):
    try:
        prefs.compute_device_type = dev_type
        for d in prefs.devices:
            d.use = True
        print(f'Cycles device enabled: {dev_type}')
        break
    except Exception:
        continue
"""
open('/content/enable_gpu.py', 'w').write(enable_gpu_script)
subprocess.run(['blender', '-b', '--python', '/content/enable_gpu.py'], check=True)

### 3. AGENT WEB 接続トンネルとワーカーの起動
次のセルを実行すると、一時HTTPS URLと接続キーが表示されます。AGENT WEBのColab画面に登録してください。

In [ ]:
# ワーカーおよび一時トンネルの起動（Protocol v2）
print('Blender Worker 待機中... (AGENT WEBと接続)')
